In [1]:
%%capture
!pip install 'kaggle-environments>=1.28'

# Orbit Wars: Visualizing Your Agent

This notebook shows how to use the **Orbit Wars Tactical Visualizer** dataset to render your episodes with some improved aesthetics, APM sparklines, and combat impact effects.

### 1. Setup
First, we add the visualizer dataset to our path.

In [2]:
import sys
from pathlib import Path
import json

# Add the dataset to sys.path so we can import the render utility
viz_path = Path('/kaggle/input/datasets/fedorbaart/orbit-wars-visualizer')
if viz_path.exists():
    sys.path.append(str(viz_path))
else:
    print("Dataset not found! Make sure fedorbaart/orbit-wars-visualizer is attached.")

from orbit_wars.core.viz.embed import render
from kaggle_environments import make

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_matching_pennies_3p
[kaggle_environments.envs.

### 2. Run and Render
Now we can run a game and use our custom `render()` function.

In [3]:
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Premium tactical render!
render(env)

### 3. Load a recorded game
You can also render pre-recorded `.json` files from your own submissions or the example provided in the dataset.

In [4]:
example_path = viz_path / 'example_episode.json'
if example_path.exists():
    with example_path.open('r') as f:
        replay = json.load(f)
    render(replay)
else:
    print("Example episode not found.")

---
## Local Usage (no Kaggle dataset required)

The cells below work **locally** using only `kaggle-environments` and your
trained `EdgePolicy` checkpoint. They use the built-in `env.render(mode='html')`
renderer and save the replay as a self-contained HTML file you can open in
any browser.

Set `CHECKPOINT` to the checkpoint you want to inspect and `OPPONENT` to any
of: `'random'`, `'starter'`, `'bully'`, `'rage'`, `'dual'`, `'prospector'`,
`'nearest_sniper'`, `'baseline'`, `'shunlite'`, `'v131_2p'`.

In [ ]:
import sys, math, json, pathlib, importlib.util, IPython.display as ipd
import torch
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent, starter_agent

# ── Config ──────────────────────────────────────────────────────────────────
CHECKPOINT = 'ppo_gnn/cache/checkpoint_ppo_edge_best.pt'
OPPONENT   = 'random'   # 'random', 'starter', or a file path
MODE       = '2p'       # '2p' or '4p'
OUT_HTML   = 'game_replay.html'  # saved next to the notebook

# ── Load model ──────────────────────────────────────────────────────────────
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from ppo_gnn.edge_policy import (
    EdgePolicy, MAX_ACTIONS, MAX_CANDIDATES, EDGE_INPUT_DIM,
    FRACTION_BUCKETS, compute_candidate_edges, solve_intercept, _travel_time,
)

ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
sd = ckpt.get('model_state_dict', ckpt)
d_model = ckpt.get('d_model', 128)
separate_critic = ckpt.get('separate_critic', True)

model = EdgePolicy(d_model=d_model, n_heads=4, n_layers=3,
                   max_actions=MAX_ACTIONS, separate_critic=separate_critic)
model_sd = model.state_dict()
filtered_sd = {k: v for k, v in sd.items()
               if k not in model_sd or v.shape == model_sd[k].shape}
model.load_state_dict(filtered_sd, strict=False)
model.eval()
print(f'Loaded checkpoint: {CHECKPOINT}')

# ── Build kaggle-compatible agent callable ────────────────────────────────
num_players = 2 if MODE == '2p' else 4

def trained_agent(obs, config=None):
    planets = obs.get('planets', [])
    if not planets:
        return []
    player = obs.get('player', 0)
    fleets = obs.get('fleets', [])
    step   = obs.get('step', 0)
    max_steps = obs.get('remainingOverageTime', 500)

    if not any(int(p[1]) == player for p in planets):
        return []

    ef, edge_indices, em, num_valid = compute_candidate_edges(
        planets=planets, fleets=fleets, player_id=player,
        num_players=num_players, step=step, max_steps=max_steps,
    )
    if num_valid == 0:
        return []

    ef = torch.nan_to_num(ef, nan=0.0, posinf=1.0, neginf=-1.0)
    with torch.no_grad():
        selected, fracs, is_noop, _, _ = model.sample_action(ef, em, deterministic=True)
    if is_noop:
        return []

    actions = []
    for cand_idx, frac_idx in zip(selected, fracs):
        if cand_idx >= num_valid:
            continue
        src_pidx = edge_indices[cand_idx, 0].item()
        tgt_pidx = edge_indices[cand_idx, 1].item()
        src_p, tgt_p = planets[src_pidx], planets[tgt_pidx]
        sx, sy = float(src_p[2]), float(src_p[3])
        tx, ty = float(tgt_p[2]), float(tgt_p[3])
        omega  = float(tgt_p[4])
        src_fleet = int(float(src_p[5]))
        ships = max(1, int(src_fleet * FRACTION_BUCKETS[frac_idx]))
        if ships < 5:
            continue
        # Compute intercept with correct travel-time physics
        ix, iy = solve_intercept(sx, sy, tx, ty, omega, ships)
        tt = _travel_time(sx, sy, ix, iy, ships)
        tgt_owner = int(tgt_p[1])
        if tgt_owner != player:
            tgt_ships = float(tgt_p[5]) + float(tgt_p[6]) * tt
            if ships < int(tgt_ships * 1.05) + 2:
                continue
        angle = math.atan2(iy - sy, ix - sx)
        actions.append([int(src_p[0]), angle, ships])
    return actions

# ── Resolve opponent ──────────────────────────────────────────────────────
OPPONENT_MAP = {'random': random_agent, 'starter': starter_agent}
if OPPONENT in OPPONENT_MAP:
    opponent = OPPONENT_MAP[OPPONENT]
else:
    spec = importlib.util.spec_from_file_location('_opp', OPPONENT)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    opponent = mod.agent

# ── Run & render ─────────────────────────────────────────────────────────
env = make('orbit_wars', debug=True)
agents = [trained_agent, opponent] if MODE == '2p' else [trained_agent] + [opponent]*3
env.run(agents)

html = env.render(mode='html')
pathlib.Path(OUT_HTML).write_text(html)
print(f'Saved to {OUT_HTML}  ({len(html)//1024} KB)')

# Display inline in the notebook
ipd.display(ipd.HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}"
    width="100%" height="650" style="border:none;"></iframe>'))
